# Pruebas unitarias

## Inicializa Spark

In [0]:
%run "./00_Init_Spark"

In [0]:
%run "./00_Init_Funcs"

## Importa librerías de testing

In [0]:
import pytest
import sys

sys.dont_write_bytecode = True

## Mock data territorios

In [0]:
# region = 13 (RM)
# comuna = 13119 (Maipu)
# provincia = 131 (Santiago)
df_schema = StructType(
    [
        StructField("region", StringType(), True),
        StructField("comuna", StringType(), True),
        StructField("provincia", StringType(), True)
    ]
)
df_data = [("13", "13119", "131")]
mock_df = spark.createDataFrame(df_data,  df_schema)


# codigo_territorial
# division_politica
# territorio
df_ct_schema = StructType(
    [
        StructField("codigo_territorial", StringType(), True),
        StructField("division_politica", StringType(), True),
        StructField("territorio", StringType(), True)
    ]
)
df_ct_data = [
    ("13", "Región", "Metropolitana de Santiago"),
    ("13119", "Comuna", "Maipú"),
    ("131", "Provincia", "Santiago")
]
mock_df_ct = spark.createDataFrame(df_ct_data, df_ct_schema)

## cruce_codigos_territoriales

In [0]:
result_ct_df = cruce_codigos_territoriales(mock_df, mock_df_ct)
result = result_ct_df.collect()[0]

assert result[0] == "Metropolitana de Santiago"
assert result[1] == "Santiago"
assert result[2] == "Maipú"

## Mock data otros

In [0]:
df_schema = StructType(
    [
        StructField("id_vivienda", IntegerType(), False),
        StructField("area", StringType(), True),
        StructField("p6_fuente_agua", StringType(), True),
        StructField("parentesco", StringType(), True)
    ]
)
df_data = [
    (123456, "1", "3", "7"),
    (7890, "2", "2", "2")
]
mock_df = spark.createDataFrame(df_data,  df_schema)


df_co_schema = StructType(
    [
        StructField("campo", StringType(), True),
        StructField("codigo", StringType(), True),
        StructField("descripcion", StringType(), True)
    ]
)

# Solo un conjunto de datos ya que son muchos registros
df_co_data = [
    ("area", "1", "Urbano"),
    ("area", "2", "Rural"),
    ("p6_fuente_agua", "2", "Pozo o noria"),
    ("p6_fuente_agua", "3", "Camión aljibe"),
    ("parentesco", "2", "Esposo/a o cónyuge"),
    ("parentesco", "7", "Hermano/a")
]
mock_df_co = spark.createDataFrame(df_co_data, df_co_schema)

## cruce_codigos_otros

In [0]:
result_co_df = cruce_codigos_otros(mock_df, mock_df_co, ["id_vivienda"])
result_collect = result_co_df.collect()
result_0 = result_collect[0]
result_1 = result_collect[1]

assert result_0[0] == "123456"
assert result_0[1] == "Urbano"
assert result_0[2] == "Camión aljibe"
assert result_0[3] == "Hermano/a"

assert result_1[0] == "7890"
assert result_1[1] == "Rural"
assert result_1[2] == "Pozo o noria"
assert result_1[3] == "Esposo/a o cónyuge"